# KPI 8 — Traffic by Device Type

**requirements.md (Web & Traffic Analytics, KPI 8):** "Traffic by Device Type: Desktop/Mobile/Tablet breakdown; device-specific conversion metrics."

This notebook creates the SQL view `b2b_ecommerce.gold.kpi_8_traffic_by_device_type` (idempotent: `CREATE OR REPLACE`). Views are catalog objects, not pipeline datasets — this notebook is **not** attached to `silver_to_gold_etl`; run the cell below once (or after any change to the fact/dim definitions).

**Definitions (state formulas):**

* **Grain:** one row per **human device type** — `desktop`, `mobile`, `tablet`, `other` (unclassifiable UA fragments), `unknown` (NULL user agent). Sessions come from `fact_web_sessions` (user-confirmed rule: same `client_ip` + `user_agent`, split where the inactivity gap exceeds 30 minutes) joined to `dim_device` on `user_agent`. **Bots are excluded** via `dim_device.is_bot` — view policy; the fact keeps bot traffic gross (28,038 bot sessions excluded). NULL-UA sessions have no `dim_device` row and surface as `unknown`.
* **Traffic measures:** `session_count`; `session_share_pct` = 100 × sessions / all human sessions (sums to 100.00); `event_count` = SUM(session `event_count`); `avg_events_per_session`; `avg_session_duration_seconds` = AVG(`duration_seconds`); `distinct_visitors` = COUNT(DISTINCT `client_ip`) — per device, **not additive across devices** (one visitor may use several); `authenticated_session_count` = sessions with a `customer_sk`.
* **Conversion rule (user-confirmed):** an order converts only if **both** sides exist — a successful (2xx) `POST /checkout/confirm` in `fact_web_logs` by the same customer **and** a real order in `fact_order_items` with `|order_created_at − event_time| ≤ 300 s` (both timestamps are Argentina local time since the silver logs parser fix). Either side missing → not counted. Counted at **order grain**: each matched order once, credited to **one device** — its closest confirm event (tie-break `log_id`; 4 orders match two confirm events each). All 6,804 successful confirm events match an order; 17 web-channel orders lack a confirm event and are correctly not conversions.
* **Conversion measures:** `converted_order_count` = matched orders credited to the device; `converted_session_count` = distinct sessions containing a chosen matched confirm event (2 sessions legitimately convert twice); `conversion_rate_pct` = ROUND(100 × `converted_session_count` / `session_count`, 2) — the visit-conversion rate per the confirmed "a visit converts only if both sides exist" rule.

**Expected totals (pre-flight verified):** 5 rows; 82,810 human sessions (110,848 − 28,038 bots); 397,738 events; 6,804 converted orders; 6,802 converted sessions.

In [0]:
%sql
CREATE OR REPLACE VIEW b2b_ecommerce.gold.kpi_8_traffic_by_device_type
TBLPROPERTIES (
  'comment' = 'KPI 8 - Traffic by device type (requirements.md: Desktop/Mobile/Tablet breakdown; device-specific conversion metrics). Grain: one row per HUMAN device type (desktop, mobile, tablet, other, unknown) - sessions from fact_web_sessions (user-confirmed rule: same client_ip + user_agent, split where the inactivity gap exceeds 30 minutes) joined to dim_device on user_agent; bots EXCLUDED via dim_device.is_bot (view policy: the fact keeps bot traffic gross, 28,038 bot sessions excluded here); sessions with NULL user_agent have no dim_device row and surface as unknown. Traffic measures: session_count; session_share_pct = 100 * session_count / all human sessions (sums to 100.00); event_count = SUM(session event_count); avg_events_per_session = event_count / session_count; avg_session_duration_seconds = AVG(duration_seconds); distinct_visitors = COUNT(DISTINCT client_ip) per device (NOT additive across devices - one visitor may use several devices); authenticated_session_count = sessions with a non-null customer_sk. Conversion (user-confirmed rule, order grain): an order converts only if BOTH sides exist - a successful (2xx) POST /checkout/confirm by the same customer AND a real order with |order_created_at - event_time| <= 300 seconds (both timestamps Argentina local time since the silver logs fix); either side missing = not counted. Each matched order counted ONCE and credited to ONE device = its closest confirm event (tie-break log_id; 4 orders match two confirm events each). converted_order_count = matched orders credited to the device; converted_session_count = distinct sessions containing a chosen matched confirm event (2 sessions convert twice, by design); conversion_rate_pct = ROUND(100 * converted_session_count / session_count, 2). Verified totals: 82,810 human sessions / 397,738 events / 6,804 converted orders / 6,802 converted sessions; all 6,804 successful confirm events match an order; 17 web-channel orders lack a confirm event (correctly not conversions).'
)
AS
WITH human_sessions AS (
  SELECT
    s.session_id,
    s.client_ip,
    s.customer_sk,
    s.event_count,
    s.duration_seconds,
    COALESCE(dd.device_type, 'unknown') AS device_type
  FROM b2b_ecommerce.gold.fact_web_sessions s
  LEFT JOIN b2b_ecommerce.gold.dim_device dd
    ON s.user_agent = dd.user_agent
  WHERE COALESCE(dd.is_bot, FALSE) = FALSE
),
confirm_events AS (
  SELECT log_id, event_time, customer_sk, user_agent, client_ip
  FROM b2b_ecommerce.gold.fact_web_logs
  WHERE method = 'POST'
    AND endpoint = '/checkout/confirm'
    AND status_code BETWEEN 200 AND 299
    AND customer_sk IS NOT NULL
),
orders AS (
  SELECT order_id, customer_sk, MIN(order_created_at) AS order_created_at
  FROM b2b_ecommerce.gold.fact_order_items
  GROUP BY order_id, customer_sk
),
matched_events AS (
  -- User-confirmed conversion rule, ORDER grain: each matched order counted
  -- once, credited to ONE device = its CLOSEST confirm event (tie-break log_id).
  SELECT order_id, log_id, event_time, user_agent, client_ip
  FROM (
    SELECT
      o.order_id,
      ce.log_id,
      ce.event_time,
      ce.user_agent,
      ce.client_ip,
      ROW_NUMBER() OVER (
        PARTITION BY o.order_id
        ORDER BY ABS(UNIX_TIMESTAMP(o.order_created_at) - UNIX_TIMESTAMP(ce.event_time)),
                 ce.log_id
      ) AS rn
    FROM orders o
    JOIN confirm_events ce
      ON ce.customer_sk = o.customer_sk
     AND ABS(UNIX_TIMESTAMP(o.order_created_at) - UNIX_TIMESTAMP(ce.event_time)) <= 300
  )
  WHERE rn = 1
),
matched_sessions AS (
  -- The session containing each chosen confirm event (null-safe UA match;
  -- every event belongs to exactly one session by construction).
  SELECT
    ms.order_id,
    ms.log_id,
    sess.session_id,
    COALESCE(dd.device_type, 'unknown') AS event_device_type
  FROM matched_events ms
  JOIN b2b_ecommerce.gold.fact_web_sessions sess
    ON sess.client_ip = ms.client_ip
   AND sess.user_agent <=> ms.user_agent
   AND ms.event_time >= sess.session_start
   AND ms.event_time <= sess.session_end
  LEFT JOIN b2b_ecommerce.gold.dim_device dd
    ON ms.user_agent = dd.user_agent
),
converted_orders AS (
  SELECT event_device_type AS device_type,
         COUNT(DISTINCT order_id) AS converted_order_count
  FROM matched_sessions
  GROUP BY event_device_type
),
converted_sessions AS (
  SELECT event_device_type AS device_type,
         COUNT(DISTINCT session_id) AS converted_session_count
  FROM matched_sessions
  GROUP BY event_device_type
)
SELECT
  hs.device_type,
  COUNT(*)                                                       AS session_count,
  ROUND(100 * COUNT(*) / SUM(COUNT(*)) OVER (), 2)              AS session_share_pct,
  SUM(hs.event_count)                                            AS event_count,
  ROUND(SUM(hs.event_count) / COUNT(*), 2)                       AS avg_events_per_session,
  ROUND(AVG(hs.duration_seconds), 1)                             AS avg_session_duration_seconds,
  COUNT(DISTINCT hs.client_ip)                                   AS distinct_visitors,
  SUM(CASE WHEN hs.customer_sk IS NOT NULL THEN 1 ELSE 0 END)    AS authenticated_session_count,
  COALESCE(co.converted_order_count, 0)                          AS converted_order_count,
  COALESCE(cs.converted_session_count, 0)                        AS converted_session_count,
  ROUND(100 * COALESCE(cs.converted_session_count, 0) / COUNT(*), 2)
                                                                  AS conversion_rate_pct
FROM human_sessions hs
LEFT JOIN converted_orders  co ON co.device_type = hs.device_type
LEFT JOIN converted_sessions cs ON cs.device_type = hs.device_type
GROUP BY
  hs.device_type,
  co.converted_order_count,
  cs.converted_session_count

In [0]:
%sql
-- Verification: shape, totals, integrity. Expected: 5 rows = 5 human device
-- types (desktop/mobile/tablet/other/unknown), 0 bot rows, SUM(session_count)
-- = 82,810 (110,848 total sessions - 28,038 bot sessions), SUM(event_count)
-- = 397,738, SUM(converted_order_count) = 6,804, SUM(converted_session_count)
-- = 6,802, share sums to 100.00, 0 null device types, 0 conversion-rate
-- inconsistencies (recomputed from stored columns), converted orders >=
-- converted sessions on every device.
SELECT
  COUNT(*)                                                                AS rows_devices,
  COUNT(DISTINCT device_type)                                              AS distinct_devices,
  SUM(CASE WHEN device_type = 'bot' THEN 1 ELSE 0 END)                     AS bot_rows,
  SUM(session_count)                                                       AS total_human_sessions,
  SUM(event_count)                                                         AS total_events,
  SUM(converted_order_count)                                               AS total_converted_orders,
  SUM(converted_session_count)                                             AS total_converted_sessions,
  ROUND(SUM(session_share_pct), 2)                                         AS sum_share_pct,
  SUM(CASE WHEN device_type IS NULL THEN 1 ELSE 0 END)                     AS null_device_rows,
  SUM(CASE WHEN ROUND(100 * converted_session_count / session_count, 2)
                <> conversion_rate_pct THEN 1 ELSE 0 END)                   AS rate_mismatches,
  SUM(CASE WHEN converted_session_count > session_count
             THEN 1 ELSE 0 END)                                            AS conv_gt_sessions,
  SUM(CASE WHEN converted_order_count < converted_session_count
             THEN 1 ELSE 0 END)                                            AS orders_lt_sessions
FROM b2b_ecommerce.gold.kpi_8_traffic_by_device_type

In [0]:
%sql
-- Verification: independently recompute every device row through a different
-- path - sessions re-derived from RAW fact_web_logs events via SQL window
-- sessionization (same user-confirmed 30-min rule, no fact_web_sessions), and
-- conversions via a MIN(struct(gap, log_id)) aggregation instead of
-- ROW_NUMBER - then compare against the stored view. Expected: every match
-- count = 5 (all devices), 0 mismatches, totals identical.
WITH ev AS (
  SELECT log_id, client_ip, user_agent, event_time
  FROM b2b_ecommerce.gold.fact_web_logs
),
lagged AS (
  SELECT e.*,
         LAG(e.event_time) OVER (
           PARTITION BY e.client_ip, e.user_agent
           ORDER BY e.event_time, e.log_id
         ) AS prev_time
  FROM ev e
),
marked AS (
  SELECT l.*,
         CASE WHEN prev_time IS NULL
               OR UNIX_TIMESTAMP(l.event_time) - UNIX_TIMESTAMP(l.prev_time) > 1800
              THEN 1 ELSE 0 END AS new_session
  FROM lagged l
),
sequenced AS (
  SELECT m.*,
         SUM(m.new_session) OVER (
           PARTITION BY m.client_ip, m.user_agent
           ORDER BY m.event_time, m.log_id
           ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
         ) AS session_seq
  FROM marked m
),
sessions2 AS (
  SELECT client_ip, user_agent, session_seq,
         MIN(event_time) AS session_start,
         MAX(event_time) AS session_end,
         COUNT(*)        AS event_count
  FROM sequenced
  GROUP BY client_ip, user_agent, session_seq
),
traffic2 AS (
  SELECT COALESCE(dd.device_type, 'unknown') AS device_type,
         COUNT(*)             AS session_count,
         SUM(s2.event_count)  AS event_count
  FROM sessions2 s2
  LEFT JOIN b2b_ecommerce.gold.dim_device dd
    ON s2.user_agent = dd.user_agent
  WHERE COALESCE(dd.is_bot, FALSE) = FALSE
  GROUP BY COALESCE(dd.device_type, 'unknown')
),
confirm_events AS (
  SELECT log_id, event_time, customer_sk, user_agent, client_ip
  FROM b2b_ecommerce.gold.fact_web_logs
  WHERE method = 'POST'
    AND endpoint = '/checkout/confirm'
    AND status_code BETWEEN 200 AND 299
    AND customer_sk IS NOT NULL
),
orders AS (
  SELECT order_id, customer_sk, MIN(order_created_at) AS order_created_at
  FROM b2b_ecommerce.gold.fact_order_items
  GROUP BY order_id, customer_sk
),
pairs AS (
  SELECT o.order_id, ce.log_id, ce.event_time, ce.user_agent, ce.client_ip,
         ABS(UNIX_TIMESTAMP(o.order_created_at) - UNIX_TIMESTAMP(ce.event_time)) AS gap_s
  FROM orders o
  JOIN confirm_events ce
    ON ce.customer_sk = o.customer_sk
   AND ABS(UNIX_TIMESTAMP(o.order_created_at) - UNIX_TIMESTAMP(ce.event_time)) <= 300
),
best AS (
  -- Closest (gap, log_id) per order via aggregation - same rule as the view,
  -- different mechanism than ROW_NUMBER.
  SELECT order_id, MIN(STRUCT(gap_s, log_id)) AS best_pick
  FROM pairs
  GROUP BY order_id
),
chosen AS (
  SELECT p.order_id, p.log_id, p.event_time, p.user_agent, p.client_ip
  FROM pairs p
  JOIN best b
    ON b.order_id = p.order_id
   AND p.log_id = b.best_pick.log_id
),
conv_orders2 AS (
  SELECT COALESCE(dd.device_type, 'unknown') AS device_type,
         COUNT(DISTINCT c.order_id) AS converted_order_count
  FROM chosen c
  LEFT JOIN b2b_ecommerce.gold.dim_device dd
    ON c.user_agent = dd.user_agent
  GROUP BY COALESCE(dd.device_type, 'unknown')
),
conv_sessions2 AS (
  SELECT COALESCE(dd.device_type, 'unknown') AS device_type,
         COUNT(DISTINCT s2.client_ip || '|' || COALESCE(s2.user_agent, '') || '|' || s2.session_seq)
           AS converted_session_count
  FROM chosen c
  JOIN sessions2 s2
    ON s2.client_ip = c.client_ip
   AND s2.user_agent <=> c.user_agent
   AND c.event_time >= s2.session_start
   AND c.event_time <= s2.session_end
  LEFT JOIN b2b_ecommerce.gold.dim_device dd
    ON c.user_agent = dd.user_agent
  GROUP BY COALESCE(dd.device_type, 'unknown')
),
recomputed AS (
  SELECT t.device_type, t.session_count, t.event_count,
         COALESCE(o.converted_order_count, 0)   AS converted_order_count,
         COALESCE(s.converted_session_count, 0) AS converted_session_count
  FROM traffic2 t
  LEFT JOIN conv_orders2  o ON o.device_type = t.device_type
  LEFT JOIN conv_sessions2 s ON s.device_type = t.device_type
),
v AS (
  SELECT device_type, session_count, event_count,
         converted_order_count, converted_session_count
  FROM b2b_ecommerce.gold.kpi_8_traffic_by_device_type
)
SELECT
  COUNT(*) AS devices_compared,
  SUM(CASE WHEN r.session_count = v.session_count THEN 1 ELSE 0 END)
    AS session_count_matches,
  SUM(CASE WHEN r.event_count = v.event_count THEN 1 ELSE 0 END)
    AS event_count_matches,
  SUM(CASE WHEN r.converted_order_count = v.converted_order_count THEN 1 ELSE 0 END)
    AS converted_order_matches,
  SUM(CASE WHEN r.converted_session_count = v.converted_session_count THEN 1 ELSE 0 END)
    AS converted_session_matches,
  SUM(r.session_count)            AS recomputed_sessions,
  SUM(r.converted_order_count)   AS recomputed_converted_orders,
  SUM(r.converted_session_count) AS recomputed_converted_sessions,
  SUM(CASE WHEN r.device_type IS NULL OR v.device_type IS NULL THEN 1 ELSE 0 END)
    AS device_set_mismatches
FROM recomputed r
FULL OUTER JOIN v ON v.device_type = r.device_type

In [0]:
%sql
-- Headline: the full device breakdown (the view is small - 5 rows).
SELECT *
FROM b2b_ecommerce.gold.kpi_8_traffic_by_device_type
ORDER BY session_count DESC